In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps
import scipy.io as sio
from scipy.linalg import expm
from tqdm import trange
from scipy.integrate import odeint

In [3]:
def check_riccati(cov, mu, yt, eps):
    B = np.array(
        [
            [0, -1],
            [1, 1],
        ]
    )
    D = np.array(
        [
            [0, 0],
            [0, 1],
        ]
    )
    # first solve for P, q
    dt = 0.00005
    M = 100

    Ps = [1/eps * cov]
    for i in range(int(np.ceil(T/dt))):
        P_update = D - Ps[-1] @ B.T - B @ Ps[-1]
        Ps += [Ps[-1] + dt * P_update]
    Ps = Ps[::M]
    Ps = Ps[1:]
    Ps.reverse()

    q_fn = lambda t: (expm(-B*t) @ mu.T).T
    

    def s_fn(Y, t, i):
        return (-np.linalg.inv(Ps[i]) @ (Y - q_fn(t)).T).T
    
    return s_fn

In [4]:
n = 2
cov = np.array(
    [
        [1, 0],
        [0, 1],
    ]
)
mu = np.array([0, 0]).reshape([1, 2])

alpha = 1
eps = 0.0001
T = 5

B = np.array(
    [
        [0, -1],
        [1, 1],
    ]
)
def rhs(y, t):
    y1, y2 = y
    dydt = [y2, -1.1*y1 - 1*y2]
    return dydt


## case 1: 
y0 = np.array([0, 0.5])
t = np.linspace(0, 5, 1001)
sols = odeint(rhs, y0, t)
yt = sols[-1, :].reshape([1, 2])


def rhs_rev(y, t):
    y1, y2 = y
    dydt = [-y2, +1*y1 + 1*y2]
    return dydt


z0 = sols[-1, :]
t = np.linspace(0, 5, 1001)
z_sols = odeint(rhs_rev, z0, t)


s_fn = check_riccati(
    cov=cov,
    mu=mu,
    yt=yt, 
    eps=eps,
)

In [5]:
yt

array([[-0.04428265,  0.01793708]])

In [6]:
s_fn(yt, T, 0)

array([[0.00046052, 0.0002221 ]])